# 🎤 BahaaAI - RVC v1.5 : Model Training & Inference

## 📺 Subscribe to [BahaaAI YouTube Channel](https://www.youtube.com/@BahaaAI) for more AI tutorials

### Last Update: 5 August 2026

**This notebook covers:**
- ✅ How to install RVC voice cloning on Google Colab for FREE
- ✅ How to train RVC voice models
- ✅ How to load trained RVC models from Google Drive
- ✅ How to resume training RVC models
- ✅ How to save trained models to Google Drive
- ✅ How to perform voice conversion (inference) with RVC models

---

## Step 1 : Prepare Files

Run this cell to download and extract the RVC runtime files from Hugging Face.

In [ ]:
#@title **Step 1 : Prepare Files**

import os
import zipfile
from IPython.display import clear_output
from ipywidgets import Button

clear_output()

# Download RVC from Hugging Face
hf_rvc_zip = "https://huggingface.co/datasets/BahaaMahmoud88/testrvc/resolve/main/RVC_without_f0Ov2Super32.zip"
zip_path = "/content/RVC.zip"
print("⏳ Downloading RVC runtime files... This may take a few minutes.")
os.system(f"wget -O {zip_path} {hf_rvc_zip} > /dev/null 2>&1")

# Validate download
if not os.path.exists(zip_path) or os.path.getsize(zip_path) < 5_000_000:
    raise FileNotFoundError("❌ RVC.zip is missing or corrupted!")

# Extract zip
extract_path = "/content/RVC"
os.makedirs(extract_path, exist_ok=True)
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

# Download f0Ov2Super32 pretrained models
print("⏳ Downloading pretrained f0Ov2Super32 models...")
os.system("wget -O /content/RVC/assets/pretrained_v2/f0Ov2Super32kD.pth https://huggingface.co/datasets/BahaaMahmoud88/testrvc/resolve/main/f0Ov2Super32kD.pth")
os.system("wget -O /content/RVC/assets/pretrained_v2/f0Ov2Super32kG.pth https://huggingface.co/datasets/BahaaMahmoud88/testrvc/resolve/main/f0Ov2Super32kG.pth")

print(f"✅ RVC extracted to {extract_path}")

# Optional: remove zip after extraction
if os.path.exists(zip_path):
    os.remove(zip_path)

clear_output()
Button(description="✔ Done 👍", button_style="success")

## Step 2 : Install RVC (5 Minutes with uv)

This step installs all the required Python packages for RVC. It takes about 5 minutes.

In [ ]:
#@title **Step 2 : Install RVC** ( 5 Minutes with uv )

import os
from IPython.display import clear_output
from ipywidgets import Button

clear_output()

print("⏳ Installing RVC dependencies... This takes about 5 minutes.")

# Install uv for faster package management
os.system("pip install uv > /dev/null 2>&1")

# Install PyTorch with uv (CUDA 12.1)
os.system("uv pip install --system torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 > /dev/null 2>&1")

# Install torchcrepe
os.system("uv pip install --system git+https://github.com/maxrmorrison/torchcrepe.git > /dev/null 2>&1")

# Install RVC requirements with uv (Python 3.12 compatible)
os.system("cd /content/RVC && uv pip install --system -r requirements-py312.txt > /dev/null 2>&1")

clear_output()
Button(description="✔ Done 👍", button_style="success")

## Step 3 : 5 August 2026 Update (Required Patch)

This patches the fairseq library to be compatible with newer PyTorch versions (weights_only loading).

In [ ]:
#@title **Step 3 : 5 August 2026 Update**

import os
from IPython.display import clear_output
from ipywidgets import Button

clear_output()

print("⏳ Applying fairseq compatibility patch...")

# Patch fairseq checkpoint_utils.py for newer PyTorch
os.system('sed -i \'s/state = torch.load(f, map_location=torch.device("cpu"))/state = torch.load(f, map_location=torch.device("cpu"), weights_only=False)/\' $(python -c "import fairseq; import os; print(os.path.join(os.path.dirname(fairseq.__file__), \'checkpoint_utils.py\'))")')

clear_output()
Button(description="✔ Done 👍", button_style="success")

---

# 📢 Training RVC Models

## Step 4 : Mount Google Drive

Connect your Google Drive to save and load trained RVC models.

In [ ]:
#@title **Step 4 : Mount Google Drive**

import os
from google.colab import drive
from IPython.display import clear_output
from ipywidgets import Button

clear_output()

# Mount Google Drive
drive.mount('/content/drive')

# Create RVC folder structure in Drive if not exists
os.makedirs('/content/drive/MyDrive/RVC/models', exist_ok=True)
os.makedirs('/content/drive/MyDrive/RVC/datasets', exist_ok=True)

print("✅ Google Drive mounted successfully!")
print("📁 Folders created: /content/drive/MyDrive/RVC/")

clear_output()
Button(description="✔ Done 👍", button_style="success")

## Step 5 : Upload Dataset (For Training New Models)

Upload your audio dataset (10+ minutes of clean voice recordings) for training a new RVC model.

**Dataset requirements:**
- 10+ minutes of clean, high-quality voice recordings
- WAV format preferred
- Single speaker only
- No background noise

In [ ]:
#@title **Step 5 : Upload Dataset**

import os
from google.colab import files
from IPython.display import clear_output
from ipywidgets import Button

clear_output()

model_name = "MyVoice"  #@param {type:"string"}

# Create dataset folder for the model
dataset_path = f"/content/dataset/{model_name}"
os.makedirs(dataset_path, exist_ok=True)

print(f"📂 Upload your audio files for model: {model_name}")
print("✅ Select multiple .wav files (10+ minutes total)")

uploaded = files.upload()

for filename, data in uploaded.items():
    filepath = os.path.join(dataset_path, filename)
    with open(filepath, 'wb') as f:
        f.write(data)
    print(f"  ✓ Saved: {filename}")

print(f"\n✅ Dataset uploaded to: {dataset_path}")
print(f"📊 Total files: {len(uploaded)}")

clear_output()
Button(description="✔ Done 👍", button_style="success")

## Step 6 : Preprocess Dataset

Extract features from your audio files to prepare them for training.

In [ ]:
#@title **Step 6 : Preprocess Dataset**

import os
from IPython.display import clear_output
from ipywidgets import Button

clear_output()

model_name = "MyVoice"  #@param {type:"string"}
sample_rate = "40k"  #@param ["32k", "40k", "48k"]

dataset_path = f"/content/dataset/{model_name}"

if not os.path.exists(dataset_path):
    raise FileNotFoundError(f"❌ Dataset not found: {dataset_path}. Run Step 5 first.")

print(f"⏳ Preprocessing dataset for model: {model_name}")
print(f"📊 Sample rate: {sample_rate}")

# Run preprocessing
cmd = f"cd /content/RVC && python infer/modules/train/preprocess.py " \
      f"/content/dataset/{model_name} {sample_rate} 8 0 " \
      f"/content/RVC/logs/{model_name} 1 0"
os.system(f'{cmd} > /dev/null 2>&1')

print("✅ Preprocessing complete!")

clear_output()
Button(description="✔ Done 👍", button_style="success")

## Step 7 : Extract Pitch (f0) Features

Extract pitch features using RMVPE algorithm.

In [ ]:
#@title **Step 7 : Extract f0 Features**

import os
from IPython.display import clear_output
from ipywidgets import Button

clear_output()

model_name = "MyVoice"  #@param {type:"string"}
f0_method = "rmvpe"  #@param ["rmvpe", "pm", "harvest", "crepe", "crepe-tiny"]

print(f"⏳ Extracting f0 features using {f0_method}...")

# Run f0 extraction
cmd = f"cd /content/RVC && python infer/modules/train/extract/extract_f0_print.py " \
      f"/content/RVC/logs/{model_name} 1 0 0 {f0_method}"
os.system(f'{cmd} > /dev/null 2>&1')

print("✅ f0 features extracted!")

clear_output()
Button(description="✔ Done 👍", button_style="success")

## Step 8 : Train RVC Model

Train the RVC model. **Recommended: 300-500 epochs** for best quality.

**Tips:**
- Total training time: ~30 minutes for 100 epochs on T4 GPU
- Save checkpoints to Google Drive every 50 epochs (in case of disconnection)
- The model improves significantly with more epochs

In [ ]:
#@title **Step 8 : Train RVC Model**

import os
from IPython.display import clear_output
from ipywidgets import Button

clear_output()

model_name = "MyVoice"  #@param {type:"string"}
total_epochs = 300  #@param {type:"integer", min:1, max:1000}
save_every_epoch = 50  #@param {type:"integer", min:1, max:100}
batch_size = 8  #@param {type:"integer", min:1, max:32}
sample_rate = "40k"  #@param ["32k", "40k", "48k"]
if_f0 = True  #@param {type:"boolean"}
if_protect = 0.5  #@param {type:"number"}

print(f"🚀 Training model: {model_name}")
print(f"📊 Total epochs: {total_epochs}")
print(f"💾 Save every: {save_every_epoch} epochs")
print(f"📦 Batch size: {batch_size}")
print(f"🎵 Sample rate: {sample_rate}")
print("\n⏳ Training started... (this may take a while)")

# Run training
cmd = (f"cd /content/RVC && python infer/modules/train/train.py "
       f"-e {model_name} "
       f"-sr {sample_rate} "
       f"-f0 {int(if_f0)} "
       f"-bs {batch_size} "
       f"-g 0 "
       f"-te {total_epochs} "
       f"-se {save_every_epoch} "
       f"-pg {if_protect} "
       f"-l 0 "
       f"-c 0")
os.system(cmd)

print("\n✅ Training complete!")

clear_output()
Button(description="✔ Done 👍", button_style="success")

## Step 9 : Generate Index File

Generate the trained feature index file for the model.

In [ ]:
#@title **Step 9 : Generate Index File**

import os
from IPython.display import clear_output
from ipywidgets import Button

clear_output()

model_name = "MyVoice"  #@param {type:"string"}

print(f"⏳ Generating index file for model: {model_name}")

# Generate index
cmd = f"cd /content/RVC && python infer/modules/train/index.py " \
      f"/content/RVC/logs/{model_name} 1"
os.system(f'{cmd} > /dev/null 2>&1')

print("✅ Index file generated!")

clear_output()
Button(description="✔ Done 👍", button_style="success")

## Step 10 : Save Trained Model to Google Drive

Save your trained model (.pth and .index files) to Google Drive so you don't lose it when the Colab session ends.

In [ ]:
#@title **Step 10 : Save Model to Google Drive**

import os
import shutil
from IPython.display import clear_output
from ipywidgets import Button

clear_output()

model_name = "MyVoice"  #@param {type:"string"}

source_pth = f"/content/RVC/weights/{model_name}.pth"
source_index = f"/content/RVC/logs/{model_name}/added_IVF512_Flat_nprobe_1_{model_name}_v2.index"

dest_folder = f"/content/drive/MyDrive/RVC/models/{model_name}"
os.makedirs(dest_folder, exist_ok=True)

dest_pth = f"{dest_folder}/{model_name}.pth"
dest_index = f"{dest_folder}/{model_name}.index"

if os.path.exists(source_pth):
    shutil.copy(source_pth, dest_pth)
    print(f"✅ Model saved: {dest_pth}")
else:
    print(f"❌ Model .pth not found: {source_pth}")

if os.path.exists(source_index):
    shutil.copy(source_index, dest_index)
    print(f"✅ Index saved: {dest_index}")
else:
    print(f"❌ Index not found: {source_index}")

print(f"\n📁 Model location: {dest_folder}")

clear_output()
Button(description="✔ Done 👍", button_style="success")

---

# 🔄 Load Trained Model from Google Drive

## Step 11 : Load Trained Model from Google Drive

If you previously trained and saved a model, you can load it here to continue training or use it for inference.

In [ ]:
#@title **Step 11 : Load Trained Model from Google Drive**

import os
import shutil
from IPython.display import clear_output
from ipywidgets import Button

clear_output()

model_name = "Bahaa AI"  #@param {type:"string"}

drive_folder = f"/content/drive/MyDrive/RVC/models/{model_name}"
dest_folder = f"/content/RVC/weights"
dest_logs = f"/content/RVC/logs"

os.makedirs(dest_folder, exist_ok=True)
os.makedirs(dest_logs, exist_ok=True)

print(f"⏳ Loading model: {model_name} from Google Drive...")

if not os.path.exists(drive_folder):
    raise FileNotFoundError(f"❌ Model not found in Drive: {drive_folder}")

# Copy .pth file
source_pth = f"{drive_folder}/{model_name}.pth"
dest_pth = f"{dest_folder}/{model_name}.pth"
if os.path.exists(source_pth):
    shutil.copy(source_pth, dest_pth)
    print(f"  ✓ Loaded: {model_name}.pth")

# Copy .index file
source_index = f"{drive_folder}/{model_name}.index"
dest_index = f"{dest_logs}/{model_name}/added_IVF512_Flat_nprobe_1_{model_name}_v2.index"
os.makedirs(os.path.dirname(dest_index), exist_ok=True)
if os.path.exists(source_index):
    shutil.copy(source_index, dest_index)
    print(f"  ✓ Loaded: {model_name}.index")

print(f"\n✅ Model '{model_name}' loaded successfully!")
print("📝 You can now continue training (Step 8) or run inference (Step 13)")

clear_output()
Button(description="✔ Done 👍", button_style="success")

## Step 12 : Resume Training (Continue from where you stopped)

Continue training the loaded model. **Example:** if you stopped at 300 epochs, you can resume up to 400 or 500 epochs to improve quality.

In [ ]:
#@title **Step 12 : Resume Training**

import os
from IPython.display import clear_output
from ipywidgets import Button

clear_output()

model_name = "Bahaa AI"  #@param {type:"string"}
total_epochs = 400  #@param {type:"integer", min:1, max:1000}
save_every_epoch = 50  #@param {type:"integer", min:1, max:100}
batch_size = 8  #@param {type:"integer", min:1, max:32}
sample_rate = "40k"  #@param ["32k", "40k", "48k"]

print(f"🔄 Resuming training: {model_name}")
print(f"📊 Target epochs: {total_epochs}")
print("\n⏳ Training resumed from last checkpoint...")

cmd = (f"cd /content/RVC && python infer/modules/train/train.py "
       f"-e {model_name} "
       f"-sr {sample_rate} "
       f"-f0 1 "
       f"-bs {batch_size} "
       f"-g 0 "
       f"-te {total_epochs} "
       f"-se {save_every_epoch} "
       f"-pg 0.5 "
       f"-l 0 "
       f"-c 0")
os.system(cmd)

print("\n✅ Resumed training complete!")
print("💾 Don't forget to save the updated model to Drive (Step 10)")

clear_output()
Button(description="✔ Done 👍", button_style="success")

---

# 🎵 RVC Inference (Voice Conversion)

## Step 13 : Upload Target Audio

Upload the audio file you want to convert to the trained voice.

In [ ]:
#@title **Step 13 : Upload Target Audio**

import os
from google.colab import files
from IPython.display import clear_output, Audio
from ipywidgets import Button

clear_output()

os.makedirs("/content/audio_input", exist_ok=True)

print("📂 Upload the audio file you want to convert")
print("✅ Supported formats: .wav, .mp3, .ogg, .flac")

uploaded = files.upload()

for filename, data in uploaded.items():
    filepath = f"/content/audio_input/{filename}"
    with open(filepath, 'wb') as f:
        f.write(data)
    print(f"  ✓ Saved: {filename}")
    
    # Show audio player
    display(Audio(filepath))

clear_output()
Button(description="✔ Done 👍", button_style="success")

## Step 14 : Run Voice Conversion (Inference)

Run inference to convert the uploaded audio using your trained RVC model.

**Pitch settings (recommended):**
- Male voice → Female: +12 semitones
- Female voice → Male: -12 semitones
- Same gender: 0 semitones

In [ ]:
#@title **Step 14 : Run Voice Conversion (Inference)**

import os
from IPython.display import clear_output, Audio
from ipywidgets import Button

clear_output()

#@markdown **Model Settings**
model_name = "Bahaa AI"  #@param {type:"string"}
input_filename = "input.wav"  #@param {type:"string"}

#@markdown **Pitch Settings**
f0_up_key = 0  #@param {type:"integer", min:-24, max:24}
f0_method = "rmvpe"  #@param ["rmvpe", "pm", "harvest"]

#@markdown **Quality Settings**
index_rate = 0.75  #@param {type:"slider", min:0, max:1, step:0.05}
filter_radius = 3  #@param {type:"integer", min:0, max:10}
resample_sr = 0  #@param {type:"integer", min:0, max:48000}
rms_mix_rate = 0.0  #@param {type:"slider", min:0, max:1, step:0.05}
protect = 0.5  #@param {type:"slider", min:0, max:0.5, step:0.05}

input_path = f"/content/audio_input/{input_filename}"
output_path = f"/content/audio_output/{os.path.splitext(input_filename)[0]}_converted.wav"
os.makedirs("/content/audio_output", exist_ok=True)

if not os.path.exists(input_path):
    raise FileNotFoundError(f"❌ Input audio not found: {input_path}")

print(f"🎵 Converting: {input_filename}")
print(f"🎤 Model: {model_name}")
print(f"🎼 Pitch shift: {f0_up_key} semitones")
print("\n⏳ Running inference...")

model_pth = f"/content/RVC/weights/{model_name}.pth"
model_index = f"/content/RVC/logs/{model_name}/added_IVF512_Flat_nprobe_1_{model_name}_v2.index"

if not os.path.exists(model_pth):
    raise FileNotFoundError(f"❌ Model not found: {model_pth}. Load model first (Step 11).")

cmd = (f"cd /content/RVC && python infer/modules/train/extract_feature_print.py "
       f"{model_pth} {input_path} {model_index} "
       f"{output_path} {f0_up_key} {f0_method} "
       f"{index_rate} {filter_radius} {resample_sr} {rms_mix_rate} {protect}")
os.system(f'{cmd} > /dev/null 2>&1')

if os.path.exists(output_path):
    print("\n✅ Conversion complete!")
    print(f"💾 Saved to: {output_path}")
    
    # Show audio player for result
    print("\n🔊 Preview:")
    display(Audio(output_path))
else:
    print("❌ Conversion failed. Check the model and audio file.")

clear_output()
Button(description="✔ Done 👍", button_style="success")

## Step 15 : Download Converted Audio

Download the converted audio file to your local machine.

In [ ]:
#@title **Step 15 : Download Converted Audio**

import os
from google.colab import files
from IPython.display import clear_output
from ipywidgets import Button

clear_output()

output_filename = "input_converted.wav"  #@param {type:"string"}
output_path = f"/content/audio_output/{output_filename}"

if os.path.exists(output_path):
    print(f"⏳ Downloading: {output_filename}")
    files.download(output_path)
    print("✅ Download started!")
else:
    print(f"❌ File not found: {output_path}")

clear_output()
Button(description="✔ Done 👍", button_style="success")

---

# 🎉 Done! You can now:

✅ Train your own RVC voice models
✅ Save models to Google Drive
✅ Load models back anytime
✅ Resume training to improve quality
✅ Convert any voice to your trained voice

## 📺 Subscribe to [BahaaAI YouTube Channel](https://www.youtube.com/@BahaaAI) for more AI tutorials!

## 🟨 Related Tutorials:
- [How to Clone Any Voice with RVC on Google Colab for FREE](https://youtu.be/VJgOxeGir4M)
- [How to Install RVC on Google Colab - Clone Any Voice](https://youtu.be/DBtvXcPYcXU)

## 🔗 Links:
- 🌐 Website: https://bahaa-ai.com
- 🟢 WhatsApp: https://whatsapp.com/channel/0029VbBtL8M9xVJmwuhVIv2j
- 🔵 Telegram: https://t.me/Bahaa_AI

**Made with ❤️ by BahaaAI**